In [1]:
import cx_Oracle
import pandas as pd
from sklearn.preprocessing import StandardScaler
from datetime import datetime
from utils import helpers
import os

### 处理谢博给的文件，转换为统一格式

In [2]:
df=pd.read_csv('input/250417stock.csv')

In [3]:
df

,CREATE_TIME,INDUSTRY_CODE
0,2024/7/8,2070008351
1,2024/7/8,2070008344
2,2024/7/8,2070008719
3,2024/7/8,2070008789
4,2024/7/8,2070008348
...,...,...
185,2025/4/14,2070008344
186,2025/4/14,2070008719
187,2025/4/14,2070008339
188,2025/4/14,2070008716


In [4]:
df['CREATE_TIME']=pd.to_datetime(df['CREATE_TIME'])
df['CREATE_TIME']=df['CREATE_TIME'].dt.strftime('%Y%m%d')

In [5]:
trading_days=helpers.get_trade_days()

In [6]:
trading_days=list(trading_days['TRADE_DAYS'])

In [7]:
trading_days

['19901203',
 '19901204',
 '19901205',
 '19901206',
 '19901207',
 '19901210',
 '19901211',
 '19901212',
 '19901213',
 '19901214',
 '19901217',
 '19901218',
 '19901219',
 '19901220',
 '19901221',
 '19901224',
 '19901225',
 '19901226',
 '19901227',
 '19901228',
 '19901231',
 '19910102',
 '19910103',
 '19910104',
 '19910107',
 '19910108',
 '19910109',
 '19910110',
 '19910111',
 '19910114',
 '19910115',
 '19910116',
 '19910117',
 '19910118',
 '19910121',
 '19910122',
 '19910123',
 '19910124',
 '19910125',
 '19910128',
 '19910129',
 '19910130',
 '19910131',
 '19910201',
 '19910204',
 '19910205',
 '19910206',
 '19910207',
 '19910208',
 '19910211',
 '19910212',
 '19910213',
 '19910214',
 '19910219',
 '19910220',
 '19910221',
 '19910222',
 '19910225',
 '19910226',
 '19910227',
 '19910228',
 '19910301',
 '19910304',
 '19910305',
 '19910306',
 '19910307',
 '19910308',
 '19910311',
 '19910312',
 '19910313',
 '19910314',
 '19910315',
 '19910318',
 '19910319',
 '19910320',
 '19910321',
 '19910322',

In [8]:
def map_to_trading_day(date):
    for td in trading_days:
        if td>=date:
            return td
    return None

In [9]:
df['TRADE_DT']=df['CREATE_TIME'].apply(map_to_trading_day)

In [10]:
df=df[['TRADE_DT','INDUSTRY_CODE']]

In [11]:
df

,TRADE_DT,INDUSTRY_CODE
0,20240708,2070008351
1,20240708,2070008344
2,20240708,2070008719
3,20240708,2070008789
4,20240708,2070008348
...,...,...
185,20250414,2070008344
186,20250414,2070008719
187,20250414,2070008339
188,20250414,2070008716


In [12]:
group_size=df.groupby('TRADE_DT').size()
df['weight']=df.groupby('TRADE_DT')['TRADE_DT'].transform(lambda x: 1/len(x))

In [13]:
df

,TRADE_DT,INDUSTRY_CODE,weight
0,20240708,2070008351,0.2
1,20240708,2070008344,0.2
2,20240708,2070008719,0.2
3,20240708,2070008789,0.2
4,20240708,2070008348,0.2
...,...,...,...
185,20250414,2070008344,0.2
186,20250414,2070008719,0.2
187,20250414,2070008339,0.2
188,20250414,2070008716,0.2


In [14]:
df.to_csv('input/250417stock_gai.csv',index=False)

### 读取调仓表txt格式文件

In [19]:
df=pd.read_csv("input/trade_log_25071501.txt",header=None,names=['date','operation','stock','price','amount','factor'])

In [20]:
df

,date,operation,stock,price,amount,factor
0,2023-01-04,buy,603998.SH,37.39,50807.436354,4.229696
1,2023-01-04,buy,600779.SH,559.40,3383.836693,6.767466
2,2023-01-04,buy,002401.SZ,95.90,19806.270079,7.204789
3,2023-01-04,buy,002826.SZ,11.24,168965.608126,1.030979
4,2023-01-04,buy,002511.SZ,111.61,17012.989560,8.158472
...,...,...,...,...,...,...
6025,2025-07-02,buy,002658.SZ,54.81,78930.840536,5.787345
6026,2025-07-02,buy,603601.SH,68.48,63173.517538,13.918807
6027,2025-07-02,buy,600990.SH,138.53,31214.025548,4.815143
6028,2025-07-02,buy,603716.SH,61.42,70422.257596,3.525874


In [21]:
df['date']=pd.to_datetime(df['date'])
df['date']=df['date'].dt.strftime('%Y%m%d')
df['stock']=df['stock'].str.strip()

In [22]:
df.shape

(6030, 6)

In [23]:
df=df[df['stock'].str.endswith(('.SH','.SZ'))]

In [24]:
df.shape

(6030, 6)

In [25]:
df

,date,operation,stock,price,amount,factor
0,20230104,buy,603998.SH,37.39,50807.436354,4.229696
1,20230104,buy,600779.SH,559.40,3383.836693,6.767466
2,20230104,buy,002401.SZ,95.90,19806.270079,7.204789
3,20230104,buy,002826.SZ,11.24,168965.608126,1.030979
4,20230104,buy,002511.SZ,111.61,17012.989560,8.158472
...,...,...,...,...,...,...
6025,20250702,buy,002658.SZ,54.81,78930.840536,5.787345
6026,20250702,buy,603601.SH,68.48,63173.517538,13.918807
6027,20250702,buy,600990.SH,138.53,31214.025548,4.815143
6028,20250702,buy,603716.SH,61.42,70422.257596,3.525874


In [19]:
10.61*252859.271662+63.59*42188.660952+23.80*112698.304132+59.42*45146.941709+43.01*62372.620184

13413111.13107572

In [21]:
41.03*45167+8.59*199158+49.11*38430+52.91*35399+15.06*121854

9159348.86

In [20]:
10.95*82900+64.42*14300+44.11*21200+60.60*15200+23.91*35000

4522063.0

In [22]:
53.45*35400+49.16*38400+44.62*44800+15.51*119100+8.58*179300

9164485.0

In [241]:
1.89*50

94.5

In [26]:
#计算权重
df['money']=df['price']*df['amount']

In [27]:
df['weight']=0

In [28]:
results=[]
all_dates=sorted(df['date'].unique())

for date in all_dates:
    group=df[df['date']==date]
    stocks=group['stock'].unique()
    
    current_day_weights={}
    
    for stock in stocks:
        stock_data=group[group['stock']==stock]
        net_value = 0
        for _,row in stock_data.iterrows():
            if row['operation']=='buy':
                net_value+=row['money']
            else:
                net_value-=row['money']
        
        if net_value>0:
            current_day_weights[stock]=net_value
            

    total_weight=sum(current_day_weights.values())
    if total_weight>0:
        normalized_weights={stock:weight/total_weight for stock,weight in current_day_weights.items()}
    else:
        normalized_weights=current_day_weights
    
    for stock,weight in normalized_weights.items():
        results.append({
            'date':date,
            'stock':stock,
            'weight':weight
        })
result_df=pd.DataFrame(results)

In [29]:
result_df

,date,stock,weight
0,20230104,603998.SH,0.020031
1,20230104,600779.SH,0.019960
2,20230104,002401.SZ,0.020028
3,20230104,002826.SZ,0.020026
4,20230104,002511.SZ,0.020022
...,...,...,...
3035,20250702,002658.SZ,0.200029
3036,20250702,603601.SH,0.200025
3037,20250702,600990.SH,0.199931
3038,20250702,603716.SH,0.199989


In [30]:
result=df[['date','stock','weight']]

In [31]:
result

,date,stock,weight
0,20230104,603998.SH,0
1,20230104,600779.SH,0
2,20230104,002401.SZ,0
3,20230104,002826.SZ,0
4,20230104,002511.SZ,0
...,...,...,...
6025,20250702,002658.SZ,0
6026,20250702,603601.SH,0
6027,20250702,600990.SH,0
6028,20250702,603716.SH,0


In [32]:
results1=pd.merge(result,result_df,on=['date','stock'],how='left')

In [33]:
results2=results1[['date','stock','weight_y']]
results2['weight_y']=results2['weight_y'].fillna(0)
results2=results2.rename(columns={'weight_y':'weight'})
results2

/tmp/ipykernel_4052194/700048785.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  results2['weight_y']=results2['weight_y'].fillna(0)


,date,stock,weight
0,20230104,603998.SH,0.020031
1,20230104,600779.SH,0.019960
2,20230104,002401.SZ,0.020028
3,20230104,002826.SZ,0.020026
4,20230104,002511.SZ,0.020022
...,...,...,...
6025,20250702,002658.SZ,0.200029
6026,20250702,603601.SH,0.200025
6027,20250702,600990.SH,0.199931
6028,20250702,603716.SH,0.199989


In [34]:
results2.to_csv('input/25071501trade_log.csv',index=False)

In [42]:
result['stock'][0]

'603176.SH'

In [87]:
0.01899*50

0.9495

In [33]:
import backtrader as bt
from datetime import datetime, timedelta
import pandas as pd
import numpy as np


# 1. 创建排序算子类
class SelectByOpenPrice:
    def __init__(self, K=100, ascending=False):
        """
        按开盘价排序选择前K只股票
        :param K: 选择的股票数量
        :param ascending: True表示从低到高排序（低价优先），False表示从高到低排序（高价优先）
        """
        self.K = K
        self.ascending = ascending

    def __call__(self, context):
        """执行排序逻辑"""
        # 获取当前策略和特征数据
        stra = context['strategy']
        features = context['features']

        # 确保有开盘价数据
        if 'open' not in features.columns:
            return

        # 获取当前日期的股票数据
        current_date = stra.datas[0].datetime.date(0)  # 获取当前日期
        current_bar = features[features.index == current_date]

        if current_bar.empty:
            return

        # 按开盘价排序
        current_bar = current_bar.sort_values('open', ascending=self.ascending)

        # 获取前K只股票
        selected_stocks = current_bar.head(self.K).index.tolist()

        # 保存到context中，供后续逻辑使用
        context['selected'] = selected_stocks
        return selected_stocks


# 2. 自定义数据类（如果需要额外字段）
class CustomData(bt.feeds.PandasData):
    lines = ('open',)  # 仅需要开盘价，其他字段可以按需添加
    params = (('open', 12),)  # 假设开盘价在CSV的第2列


# 3. 策略类
class MultiStockStrategy(bt.Strategy):
    params = (
        ('K', 3),  # 选择的股票数量
        ('ascending', False),  # True: 低价优先，False: 高价优先
        ('min_period', 1),  # 用于计算指标的最小周期
    )

    def __init__(self):
        # 初始化排序算子
        self.select = SelectByOpenPrice(K=self.params.K, ascending=self.params.ascending)

        # 为每只股票创建SMA20指标
        self.smas = {}
        for i, d in enumerate(self.datas):
            # 仅对股票数据计算SMA20
            if i > 0:  # 假设第0个数据是指数
                self.smas[d] = bt.ind.SMA(d, period=self.params.min_period)

        # 用于存储选择的股票
        self.selected_stocks = []
        self.selected_stocks_set = set()

        # 用于跟踪已持仓的股票
        self.holdings = set()

        # 用于存储特征数据
        self.features = None

    def next(self):
        #获取当前日期
        current_date=self.datas[0].datetime.date(0)
        print(f"current_date:{current_date}")
        stock_list = ['000001.SZ', '000002.SZ', '688355.SH']
        stock_prices=[]
        # if current_date=="2021-01-07":
        for stock in stock_list:
            stock_data = self.getdatabyname(stock)
            close_price = stock_data.close[0]
            stock_prices.append((stock, close_price))
        print(stock_prices)
        self.sorted_stocks = sorted(stock_prices, key=lambda x: x[1])


        # 8. 记录日志
        print(f'已持仓: {list(self.holdings)}, 选择股票: {self.selected_stocks}')

    def prenext(self):
        """必须跳过最小周期前的数据"""
        self.next()  # 直接跳转到next，避免最小周期问题


# 4. 主程序
if __name__ == '__main__':
    # 创建Cerebro引擎
    cerebro = bt.Cerebro()

    # 设置初始资金
    cerebro.broker.setcash(10000000.0)

    # 添加佣金模式（考虑中国股市）
    cerebro.broker.setcommission(commission=0.001, margin=None, mult=1.0)

    # 添加数据（假设已有数据文件）
    # 这里需要为每只股票添加数据
    # 实际应用中，应从CSV或数据库加载多只股票数据
    stock_list = ['000001.SZ', '000002.SZ', '688355.SH']  # 示例股票列表

    for stock in stock_list:
        # 加载数据文件，假设CSV格式为：日期,开盘价,最高价,最低价,收盘价,成交量
        data = bt.feeds.GenericCSVData(
            dataname=f'/home/quant/data_test/csv_data/daily_data/{stock}.csv',
            dtformat='%Y-%m-%d',
            datetime=1,
            open=12,
            high=13,
            low=14,
            close=15,
            volume=9,
            fromdate=datetime(2020, 1, 1),
            todate=datetime(2023, 12, 31)
        )
        cerebro.adddata(data,name=stock)
        import backtrader as bt
from datetime import datetime, timedelta
import pandas as pd
import numpy as np


# 1. 创建排序算子类
class SelectByOpenPrice:
    def __init__(self, K=100, ascending=False):
        """
        按开盘价排序选择前K只股票
        :param K: 选择的股票数量
        :param ascending: True表示从低到高排序（低价优先），False表示从高到低排序（高价优先）
        """
        self.K = K
        self.ascending = ascending

    def __call__(self, context):
        """执行排序逻辑"""
        # 获取当前策略和特征数据
        stra = context['strategy']
        features = context['features']

        # 确保有开盘价数据
        if 'open' not in features.columns:
            return

        # 获取当前日期的股票数据
        current_date = stra.datas[0].datetime.date(0)  # 获取当前日期
        current_bar = features[features.index == current_date]

        if current_bar.empty:
            return

        # 按开盘价排序
        current_bar = current_bar.sort_values('open', ascending=self.ascending)

        # 获取前K只股票
        selected_stocks = current_bar.head(self.K).index.tolist()

        # 保存到context中，供后续逻辑使用
        context['selected'] = selected_stocks
        return selected_stocks


# 2. 自定义数据类（如果需要额外字段）
class CustomData(bt.feeds.PandasData):
    lines = ('open',)  # 仅需要开盘价，其他字段可以按需添加
    params = (('open', 12),)  # 假设开盘价在CSV的第2列


# 3. 策略类
class MultiStockStrategy(bt.Strategy):
    params = (
        ('K', 3),  # 选择的股票数量
        ('ascending', False),  # True: 低价优先，False: 高价优先
        ('min_period', 1),  # 用于计算指标的最小周期
    )

    def __init__(self):
        # 初始化排序算子
        self.select = SelectByOpenPrice(K=self.params.K, ascending=self.params.ascending)

        # 为每只股票创建SMA20指标
        self.smas = {}
        for i, d in enumerate(self.datas):
            # 仅对股票数据计算SMA20
            if i > 0:  # 假设第0个数据是指数
                self.smas[d] = bt.ind.SMA(d, period=self.params.min_period)

        # 用于存储选择的股票
        self.selected_stocks = []
        self.selected_stocks_set = set()

        # 用于跟踪已持仓的股票
        self.holdings = set()

        # 用于存储特征数据
        self.features = None

    def next(self):
        #获取当前日期
        current_date=self.datas[0].datetime.date(0)
        print(f"current_date:{current_date}")
        stock_list = ['000001.SZ', '000002.SZ', '688355.SH']
        stock_prices=[]
        # if current_date=="2021-01-07":
        for stock in stock_list:
            stock_data = self.getdatabyname(stock)
            close_price = stock_data.close[0]
            stock_prices.append((stock, close_price))
        print(stock_prices)
        self.sorted_stocks = sorted(stock_prices, key=lambda x: x[1])


        # 8. 记录日志
        print(f'已持仓: {list(self.holdings)}, 选择股票: {self.selected_stocks}')

    def prenext(self):
        """必须跳过最小周期前的数据"""
        self.next()  # 直接跳转到next，避免最小周期问题


# 4. 主程序
if __name__ == '__main__':
    # 创建Cerebro引擎
    cerebro = bt.Cerebro()

    # 设置初始资金
    cerebro.broker.setcash(10000000.0)

    # 添加佣金模式（考虑中国股市）
    cerebro.broker.setcommission(commission=0.001, margin=None, mult=1.0)

    # 添加数据（假设已有数据文件）
    # 这里需要为每只股票添加数据
    # 实际应用中，应从CSV或数据库加载多只股票数据
    stock_list = ['000001.SZ', '000002.SZ', '688355.SH']  # 示例股票列表

    for stock in stock_list:
        # 加载数据文件，假设CSV格式为：日期,开盘价,最高价,最低价,收盘价,成交量
        data = bt.feeds.GenericCSVData(
            dataname=f'/home/quant/data_test/csv_data/daily_data/{stock}.csv',
            dtformat='%Y-%m-%d',
            datetime=1,
            open=12,
            high=13,
            low=14,
            close=15,
            volume=9,
            fromdate=datetime(2020, 1, 1),
            todate=datetime(2023, 12, 31)
        )
        cerebro.adddata(data,name=stock)
        if stock in ['000002.SZ','000001.SZ']:
        # if stock in ['000002.SZ']:
            data.plotinfo.plot=False

    # 添加策略
    cerebro.addstrategy(MultiStockStrategy)

    # 运行回测
    print('初始资金: %.2f' % cerebro.broker.getvalue())
    cerebro.run()
    print('最终资金: %.2f' % cerebro.broker.getvalue())


初始资金: 10000000.00
current_date:2020-01-02
[('000001.SZ', 1130.511), ('000002.SZ', 4832.29), ('688355.SH', 20.3153)]
已持仓: [], 选择股票: []
current_date:2020-01-03
[('000001.SZ', 1151.285), ('000002.SZ', 4756.6), ('688355.SH', 20.3153)]
已持仓: [], 选择股票: []
current_date:2020-01-06
[('000001.SZ', 1143.9136), ('000002.SZ', 4676.46), ('688355.SH', 20.3153)]
已持仓: [], 选择股票: []
current_date:2020-01-07
[('000001.SZ', 1149.2746), ('000002.SZ', 4713.56), ('688355.SH', 20.3153)]
已持仓: [], 选择股票: []
current_date:2020-01-08
[('000001.SZ', 1116.4382), ('000002.SZ', 4701.69), ('688355.SH', 20.3153)]
已持仓: [], 选择股票: []
current_date:2020-01-09
[('000001.SZ', 1125.1499), ('000002.SZ', 4778.86), ('688355.SH', 20.3153)]
已持仓: [], 选择股票: []
current_date:2020-01-10
[('000001.SZ', 1118.4486), ('000002.SZ', 4669.04), ('688355.SH', 20.3153)]
已持仓: [], 选择股票: []
current_date:2020-01-13
[('000001.SZ', 1138.5526), ('000002.SZ', 4701.69), ('688355.SH', 20.3153)]
已持仓: [], 选择股票: []
current_date:2020-01-14
[('000001.SZ', 1123.1395)

In [34]:
figs=cerebro.plot()

<IPython.core.display.Javascript object>

In [36]:
import akshare as ak
import pandas as pd
import datetime

# 获取沪深300指数数据
def get_hs300_data(start_date, end_date):
    """
    获取沪深300指数数据
    :param start_date: 开始日期，格式'YYYY-MM-DD'
    :param end_date: 结束日期，格式'YYYY-MM-DD'
    :return: DataFrame格式的沪深300指数数据
    """
    # 获取沪深300指数数据
    hs300_data = ak.index_zh_a_hist(
        symbol="000300", 
        period="daily", 
        start_date=start_date, 
        end_date=end_date
    )
    
    # 重命名列以符合Backtrader要求
    hs300_data = hs300_data.rename(columns={
        '日期': 'date',
        '开盘': 'open',
        '最高': 'high',
        '最低': 'low',
        '收盘': 'close',
        '成交量': 'volume'
    })
    
    # 转换日期格式
    hs300_data['date'] = pd.to_datetime(hs300_data['date'])
    hs300_data = hs300_data.set_index('date')
    
    # 添加openinterest列（Backtrader要求）
    hs300_data['openinterest'] = 0
    
    # 选择Backtrader需要的列
    hs300_data = hs300_data[['open', 'high', 'low', 'close', 'volume', 'openinterest']]
    
    return hs300_data

In [37]:
class BaselineStrategy(bt.Strategy):
    def __init__(self):
        self.dataclose = self.datas[0].close
        self.bought = False
        self.buyprice = None

    def next(self):
        # 如果尚未买入，且不是回测的第一天
        if not self.bought:
            # 在第一个交易日买入
            self.buy()
            self.bought = True
            self.buyprice = self.dataclose[0]
        
        # 如果是回测的最后一天，卖出
        if self.bought and self.datas[0].datetime.date(0) == self.datas[0].datetime.date(-1):
            self.sell()

In [41]:
import backtrader as bt
import datetime

# 1. 获取沪深300指数数据
start_date = "2015-01-01"
end_date = "2023-12-31"
hs300_data = get_hs300_data(start_date, end_date)

# # 2. 创建回测引擎
# cerebro = bt.Cerebro()
# 
# # 3. 添加沪深300指数数据作为基线
# data_hs300 = bt.feeds.PandasData(
#     dataname=hs300_data,
#     fromdate=datetime.datetime.strptime(start_date, "%Y-%m-%d"),
#     todate=datetime.datetime.strptime(end_date, "%Y-%m-%d")
# )
# cerebro.adddata(data_hs300, name='HS300')
# 
# # 4. 添加您的自定义策略
# # 例如，这里添加一个简单的单均线策略
# class SimpleMAStrategy(bt.Strategy):
#     params = (('period', 20),)
#     
#     def __init__(self):
#         self.sma = bt.indicators.SimpleMovingAverage(
#             self.datas[0].close, period=self.params.period)
#     
#     def next(self):
#         if not self.position:
#             if self.data.close[0] > self.sma[0]:
#                 self.buy()
#         elif self.data.close[0] < self.sma[0]:
#             self.sell()
# 
# cerebro.addstrategy(SimpleMAStrategy)
# 
# # 5. 添加基线策略（买入并持有）
# cerebro.addstrategy(BaselineStrategy)
# 
# # 6. 设置初始资金
# cerebro.broker.setcash(100000.0)
# 
# # 7. 运行回测
# print('初始资金: %.2f' % cerebro.broker.getvalue())
# results = cerebro.run()
# print('最终资金: %.2f' % cerebro.broker.getvalue())
# 
# # 8. 绘制结果

  0%|          | 0/16 [00:00<?, ?it/s]

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

In [ ]:
cerebro.plot(style='candlestick')